<a href="https://colab.research.google.com/github/roshia3/nvda-ml-forecasting/blob/main/NVDA_Stock_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance -q

In [3]:
import yfinance as yf
import pandas as pd

# Download NVIDIA historical stock data
nvda = yf.download(
    "NVDA",
    start="2020-01-01",
    end="2026-01-01",
    interval="1d",
    auto_adjust=True,
    progress=False
)

# Display the first 5 rows
nvda.head()

Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2020-01-02,5.957137,5.957137,5.877927,5.928333,237536000
2020-01-03,5.861786,5.905489,5.812870,5.837701,205384000
2020-01-06,5.886370,5.891584,5.742600,5.768672,262636000
2020-01-07,5.957635,6.003323,5.869734,5.914678,314856000
2020-01-08,5.968811,6.010030,5.913438,5.953415,277108000


In [4]:
print("Dataset shape:", nvda.shape)
print("Columns:", nvda.columns)
print("Missing values:")
print(nvda.isnull().sum())

# Save the dataset as a CSV file
nvda.to_csv("NVDA_raw_data.csv")

Dataset shape: (1508, 5)
Columns: MultiIndex([( 'Close', 'NVDA'),
            (  'High', 'NVDA'),
            (   'Low', 'NVDA'),
            (  'Open', 'NVDA'),
            ('Volume', 'NVDA')],
           names=['Price', 'Ticker'])
Missing values:
Price   Ticker
Close   NVDA      0
High    NVDA      0
Low     NVDA      0
Open    NVDA      0
Volume  NVDA      0
dtype: int64


In [5]:

# Check for duplicate dates
print("Duplicate dates:", nvda.index.duplicated().sum())

# Check for missing values
print("\nMissing values:")
print(nvda.isnull().sum())

# Check the date range
print("\nFirst date:", nvda.index.min())
print("Last date:", nvda.index.max())

# Check dataset dimensions
print("\nDataset shape:", nvda.shape)


Duplicate dates: 0

Missing values:
Price   Ticker
Close   NVDA      0
High    NVDA      0
Low     NVDA      0
Open    NVDA      0
Volume  NVDA      0
dtype: int64

First date: 2020-01-02 00:00:00
Last date: 2025-12-31 00:00:00

Dataset shape: (1508, 5)


In [7]:

# Create a copy of the original dataset
df = nvda.copy()

# Sort data chronologically
df = df.sort_index()

# Remove duplicate dates
df = df[~df.index.duplicated(keep="first")]

# Remove rows with missing raw data
df = df.dropna()

print("Cleaned dataset shape:", df.shape)
print(df.head())


Cleaned dataset shape: (1508, 5)
Price          Close      High       Low      Open     Volume
Ticker          NVDA      NVDA      NVDA      NVDA       NVDA
Date                                                         
2020-01-02  5.957137  5.957137  5.877927  5.928333  237536000
2020-01-03  5.861786  5.905489  5.812870  5.837701  205384000
2020-01-06  5.886370  5.891584  5.742600  5.768672  262636000
2020-01-07  5.957635  6.003323  5.869734  5.914678  314856000
2020-01-08  5.968811  6.010030  5.913438  5.953415  277108000


In [9]:

# Flatten yfinance column names if necessary
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Feature 1: Daily Return
df["Daily_Return"] = df["Close"].pct_change()

# Feature 2: Trading Volume
df["Trading_Volume"] = df["Volume"]

# Feature 3: 5-Day Momentum
df["Momentum_5D"] = df["Close"].pct_change(periods=5)

# Feature 4: 5-Day Volatility
df["Volatility_5D"] = df["Daily_Return"].rolling(window=5).std()

# Display the calculated features
features = [
    "Daily_Return",
    "Trading_Volume",
    "Momentum_5D",
    "Volatility_5D"
]

print(df[features].head(10))


Price       Daily_Return  Trading_Volume  Momentum_5D  Volatility_5D
Date                                                                
2020-01-02           NaN       237536000          NaN            NaN
2020-01-03     -0.016006       205384000          NaN            NaN
2020-01-06      0.004194       262636000          NaN            NaN
2020-01-07      0.012107       314856000          NaN            NaN
2020-01-08      0.001876       277108000          NaN            NaN
2020-01-09      0.010982       255112000     0.012963       0.011290
2020-01-10      0.005350       316296000     0.034948       0.004437
2020-01-13      0.031352       319840000     0.062938       0.011421
2020-01-14     -0.018653       359088000     0.030634       0.017976
2020-01-15     -0.006915       263104000     0.021590       0.018908


In [10]:

# Calculate next trading day's closing price
df["Next_Day_Close"] = df["Close"].shift(-1)

# Create binary target labels
df["Target"] = (
    df["Next_Day_Close"] > df["Close"]
).astype(int)

# Remove rows with missing features or next-day prices
df = df.dropna(subset=features + ["Next_Day_Close"])

# Display the processed observations
print(df[features + ["Target"]].head(10))

# Check target distribution
print("\nTarget distribution:")
print(df["Target"].value_counts())


Price       Daily_Return  Trading_Volume  Momentum_5D  Volatility_5D  Target
Date                                                                        
2020-01-09      0.010982       255112000     0.012963       0.011290       1
2020-01-10      0.005350       316296000     0.034948       0.004437       1
2020-01-13      0.031352       319840000     0.062938       0.011421       0
2020-01-14     -0.018653       359088000     0.030634       0.017976       0
2020-01-15     -0.006915       263104000     0.021590       0.018908       1
2020-01-16      0.013682       284192000     0.024319       0.019179       1
2020-01-17      0.001406       253792000     0.020301       0.019240       0
2020-01-21     -0.005375       217916000    -0.016033       0.011870       1
2020-01-22      0.008752       239240000     0.011445       0.008886       1
2020-01-23      0.010995       244516000     0.029686       0.007778       0

Target distribution:
Target
1    816
0    686
Name: count, dtype: int64


In [11]:

# Select final machine learning columns
processed_df = df[features + ["Target"]].copy()

# Save processed dataset
processed_df.to_csv("NVDA_processed_data.csv")

print("Processed dataset saved successfully!")
print("Dataset shape:", processed_df.shape)
print(processed_df.head())


Processed dataset saved successfully!
Dataset shape: (1502, 5)
Price       Daily_Return  Trading_Volume  Momentum_5D  Volatility_5D  Target
Date                                                                        
2020-01-09      0.010982       255112000     0.012963       0.011290       1
2020-01-10      0.005350       316296000     0.034948       0.004437       1
2020-01-13      0.031352       319840000     0.062938       0.011421       0
2020-01-14     -0.018653       359088000     0.030634       0.017976       0
2020-01-15     -0.006915       263104000     0.021590       0.018908       1


In [12]:

# Separate input features and target
X = processed_df[features]
y = processed_df["Target"]

# Calculate 80% training split
split_index = int(len(processed_df) * 0.8)

# Split chronologically with a one-day boundary gap
X_train = X.iloc[:split_index - 1]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index - 1]
y_test = y.iloc[split_index:]

# Display dataset sizes
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining period:")
print(X_train.index.min(), "to", X_train.index.max())

print("\nTesting period:")
print(X_test.index.min(), "to", X_test.index.max())


Training samples: 1200
Testing samples: 301

Training period:
2020-01-09 00:00:00 to 2024-10-15 00:00:00

Testing period:
2024-10-17 00:00:00 to 2025-12-30 00:00:00
